# 01 — Download data

Downloads the raw data this simulation depends on:

1. The drivable road network for the configured **study area** via **OSMnx**.
2. Public charger locations around central Warsaw via **Open Charge Map** (OCM).

The conceptual focus of the thesis remains **Śródmieście**, but the *technical* study area defaults to **central Warsaw** (Śródmieście, Wola, Ochota, Mokotów, Praga-Północ) and the OCM radius is widened to 10 km, so the model has enough chargers and graph nodes for meaningful scenario comparison. See `src/config.py` (`STUDY_AREA_MODE`, `OCM_QUERY_RADIUS_KM`).

If you do not have an OCM API key, the loader will fall back to `data/raw/chargers_manual.csv`. Instructions are printed automatically.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

from src import config
from src import graph_utils
from src import charger_data

print(f'STUDY_AREA_MODE     = {config.STUDY_AREA_MODE}')
print(f'study_area_query()  = {config.study_area_query()}')
print(f'OCM centre          = ({config.WARSAW_CENTER_LAT}, {config.WARSAW_CENTER_LON})')
print(f'OCM radius (km)     = {config.OCM_QUERY_RADIUS_KM}')
print(f'OCM max results     = {config.OCM_MAX_RESULTS}')
print(f'raw CSV target      = {config.RAW_CHARGER_CSV}')

## 1. Road network (OSMnx)

In [ ]:
G = graph_utils.get_or_build_graph(force_download=False)
print(f'nodes: {G.number_of_nodes():,}  edges: {G.number_of_edges():,}')
print(f'pickled to: {config.GRAPH_PICKLE}')

## 2. Charger inventory (Open Charge Map → CSV)

Saves the wider central-Warsaw inventory to `data/raw/chargers_openchargemap_warsaw_wide.csv`. The `clean_chargers` step in notebook 02 prints a per-step funnel (raw → cleaned → snapped → boundary-filtered → aggregated).

In [ ]:
if not config.get_ocm_api_key():
    print(config.ocm_key_help_message())

df_raw = charger_data.load_chargers(prefer_cached=False)
df_raw.head()

In [ ]:
print(f'raw rows fetched: {len(df_raw):,}')
print(f'saved to:         {config.RAW_CHARGER_CSV}')